## Topic 8: Embedding Model Migration

### Concept, Intuition, Why It Exists

- Every vector stored in Qdrant is the output of one specific embedding model. That model defines a geometric space — a particular arrangement of 384 (or 768, or 1536) dimensions where "similar meaning" maps to "nearby direction." Every query must be embedded with the exact same model to land in the same geometric space as the stored vectors. If it doesn't, the query vector is in a completely different space and the similarity scores become meaningless — not slightly wrong, fundamentally broken.
- **Embedding model migration** is what happens when you need to swap the model producing those vectors: upgrading to a better model, switching from a multilingual model to a domain-specific one, moving from a local model to an API-based one, or responding to a model being deprecated. The core problem is that you can't mix old and new vectors in the same collection — they don't share a geometric space, so a query embedded with the new model will produce nonsense similarity scores against vectors produced by the old model, even though the chunks themselves haven't changed at all.
- This is named explicitly on this project's cheatsheet as a common production failure, and it shows up in two forms: an intentional migration (you decide to upgrade) and an accidental migration (a library update silently changes how the model encodes text, or someone swaps the model constant in the code without realising that every existing vector is now stale).
- This project's current model is `paraphrase-multilingual-MiniLM-L12-v2` (384 dimensions). Everything in this topic is about what happens when that model changes and what re-indexing actually involves.

### Internal Working

**Why vectors from two different models are incomparable:**
- Every embedding model learns its own mapping from text to vector space during training — the 384 numbers produced by Model A for the sentence "premature withdrawal penalty" point in a completely different direction in a completely different space than the 384 numbers Model B produces for the same sentence. "Direction" in one model's space has no relationship to "direction" in another model's space.
- Concretely: if you embed a query with Model B and search a collection full of vectors produced by Model A, the dot products you compute measure nothing meaningful — you're comparing directions in two unrelated spaces. The result is effectively random similarity scores.

**What re-indexing actually involves:**
1. Load the new embedding model.
2. Re-embed every chunk in the corpus using the new model — the chunks themselves haven't changed, only their vector representations need to change.
3. Create a new Qdrant collection with the new vector size (if the new model has a different output dimension, which is common — e.g. upgrading from a 384-dim model to a 768-dim model).
4. Upsert all re-embedded chunks into the new collection.
5. Validate that the new collection produces correct retrieval results for a set of known queries.
6. Swap the live retrieval path to point at the new collection — Qdrant's collection aliases make this a single API call.
7. Decommission the old collection.

**Why the manifest doesn't help here:**
- The incremental ingestion manifest (from the ingestion chapter) tracks content hashes of source files. A model swap doesn't change any source file — the PDFs, CSVs, and JSON files are identical. The manifest sees no changes and reports "nothing to ingest," while every stored vector is silently stale relative to the new model. The manifest must also track the embedding model name and version to detect this case.

### How It's Implemented In This Project

- This project uses `paraphrase-multilingual-MiniLM-L12-v2` (384 dimensions) everywhere — at ingest time in all the chunking/upsert code, and at query time in all the search functions. The model name is a constant defined once and shared, not duplicated per function.
- The code below demonstrates the actual migration: build a collection with the old model, prove search works, then swap to a new model and show that searching with the new model against old vectors produces wrong results — before showing the correct fix (re-embed everything with the new model into a new collection).
- For demonstration purposes, two different sentence-transformer models serve as "old" and "new" — in a real migration the models would be meaningfully different (e.g. a generic multilingual model vs. a finance-domain fine-tuned one).

### Real-World Issues, Edge Cases, Debugging

- **The failure is silent by default**: searching with a new model against old vectors throws no error. It returns results with plausible-looking scores. The only signal that something is wrong is that retrieval quality has degraded — and that's a subtle, easy-to-miss signal unless you're actively monitoring retrieval quality metrics.
- **Dimension mismatch is the loud failure**: if the new model has a different output dimension than the old one (e.g. switching from 384-dim to 768-dim), trying to upsert new vectors into the old collection immediately raises a dimension mismatch error. This is the loud, obvious failure — actually easier to catch than the silent same-dimension-different-model case.
- **Partial migration is worse than no migration**: if some chunks are re-embedded with the new model and some remain as old-model vectors in the same collection, every query — regardless of which model it uses — will get correct results for some chunks and random scores for others. This is harder to debug than either "everything is old" or "everything is new."
- **The manifest blind spot**: this project's manifest tracks content hashes only. Swapping the embedding model while leaving the manifest unchanged means the manifest reports "nothing to ingest" and the pipeline happily serves stale vectors indefinitely. The fix is tracking `embed_model_name` in the manifest alongside the content hash.

### Design Decisions & Trade-offs

- New collection for the migration vs. updating in place: always use a new collection. Updating vectors in place in the same collection means there's a window during migration where old and new vectors coexist — partial migration is worse than no migration. A new collection is built entirely with the new model, validated, then swapped in atomically.
- Staging collection + alias swap vs. direct cutover: the staging + alias pattern (build the new collection while the old one still serves traffic, then swap the alias) gives zero downtime and a clean rollback path (swap the alias back). Direct cutover means downtime while the new collection is being built, which is unacceptable for a live system.
- Re-embedding in batches vs. one chunk at a time: always batch. Re-embedding 10,000 chunks one at a time means 10,000 separate model calls; batching means one call per batch of N chunks. The embedding model's `encode()` call is designed for batches — using it one-at-a-time is the same mistake as making one API call per row instead of per batch, from the ingestion chapter.

### Alternatives & When To Use Each

- Full re-index (re-embed everything): always required when the model changes. There is no partial fix — either all vectors in a collection come from the same model, or the collection is corrupted for retrieval purposes.
- Build-then-swap with Qdrant aliases: the correct production pattern for zero downtime — build a staging collection, validate, swap alias, decommission old.
- Track model version in the manifest: the fix for the manifest blind spot — add `embed_model_name` to every manifest entry so a model change triggers a full re-embed even though source content is unchanged.
- Canary validation before swap: embed a small representative set of queries, compare top-k results from old and new collections, confirm the new collection actually retrieves better before committing to the swap.

### Common Mistakes & Production Failures

- Swapping the embedding model constant in code without rebuilding the collection — silently corrupted retrieval, no error, no obvious signal until someone notices answers are wrong.
- Migrating the model and changing chunk size or source content at the same time — impossible to tell whether a retrieval quality change came from the model swap or the content change.
- Not adding the embedding model name to the manifest, leaving the pipeline blind to model changes that require a full re-embed.
- Updating the collection in place (partial migration) rather than building a new collection with all chunks re-embedded before swapping.

### Lead-Level Interview Questions

**Q: You swap the embedding model and forget to re-index. What actually happens at query time and how would you detect it?**
A: No error is thrown. The query gets embedded with the new model, producing a vector in the new model's space. That vector gets compared via dot product to stored vectors in the old model's space. The scores are not zero — they're plausible-looking numbers in the range cosine similarity usually produces — but they measure nothing meaningful. Detection requires monitoring retrieval quality: track a known set of queries with known correct answers and alert if top-k accuracy drops. A sudden drop right after a deployment that included a model name change is the signal. Without active monitoring, this can degrade silently for days.

**Q: What is the correct zero-downtime migration procedure, step by step?**
A: Build a staging collection with the new model name and vector size. Re-embed all chunks using the new model and upsert into staging — do not touch the live collection at all during this step. Run canary validation: embed a representative set of queries with the new model and check top-k results in staging against expected answers. If validation passes, use Qdrant's alias API to atomically point the live alias from the old collection to the new staging collection — one API call, instantaneous. Decommission the old collection after confirming the new one is serving traffic correctly. If validation fails, nothing in production was touched and rollback is free.

**Q: The manifest currently tracks content hashes. A teammate upgrades the embedding model. How does the manifest behave and what is the fix?**
A: The manifest sees no changes — source file content hashes are identical, so it reports "nothing to ingest." Every existing vector silently becomes stale relative to the new model, and the pipeline serves degraded retrieval with no signal that anything is wrong. The fix is adding `embed_model_name` as a tracked field alongside the content hash in every manifest entry. On each ingestion run, if the configured model name differs from what the manifest recorded for a file, that file is treated as needing re-embedding even though its content hasn't changed.

**Q: Can you ever mix vectors from two different models in the same Qdrant collection?**
A: No — and the reason is geometric. Each model defines its own vector space. A dot product between a vector from Model A's space and a vector from Model B's space doesn't measure semantic similarity; it measures an arbitrary number that has no meaningful interpretation. The only correct answer is one model per collection, always. If you need to support multiple models for different use cases, they get separate collections, not a shared one.

### Hidden Concepts Worth Knowing

- **Model versioning is different from model naming**: the same model name can produce slightly different vectors across library versions if the tokenizer, pooling method, or normalisation behavior changed in a library update. Tracking the model name alone isn't enough — tracking the model name plus the library version (sentence-transformers version, or a pinned model hash) gives a complete picture of what produced a given set of vectors.
- **Fine-tuned models and domain shift**: upgrading from a generic model to a domain-specific fine-tuned model (e.g. a model fine-tuned on financial text) can dramatically improve retrieval quality for domain-specific queries but may degrade retrieval on general queries that the generic model handled well. A canary validation set should include both domain-specific and general queries to catch this regression before committing to the swap.

### Revision Summary

> Embedding model migration requires re-embedding every chunk in the corpus — there is no partial fix, because mixing vectors from two different models in one collection makes similarity scores meaningless. The correct production pattern is build-then-swap: build a staging collection with all chunks re-embedded using the new model, validate with known queries, then atomically swap the live alias to the new collection. The incremental manifest is blind to model changes by default — it must also track the embedding model name so a model swap triggers a full re-embed even though source content is unchanged.

In [1]:
"""
qdrant_model_migration.py
---------------------------
Demonstrates what actually happens during an embedding model migration,
and what the correct migration procedure looks like.

Shows:
  1. Build a collection with OLD model -- search works correctly
  2. Search with NEW model against OLD vectors -- silent wrong results
  3. Correct fix: re-embed everything with NEW model into NEW collection
  4. Build-then-swap with collection aliases -- zero downtime pattern
  5. Manifest blind spot -- model name not tracked, migration goes undetected

Uses :memory: mode -- no Docker required.
Install: pip install qdrant-client sentence-transformers
"""

import hashlib
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
)
from sentence_transformers import SentenceTransformer

# old model: 384 dimensions -- what this project currently uses
OLD_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
OLD_VECTOR_SIZE = 384

# new model: 384 dimensions too -- in a real migration this might differ
# using all-MiniLM-L6-v2 as the "new" model for demonstration
NEW_MODEL_NAME = "all-MiniLM-L6-v2"
NEW_VECTOR_SIZE = 384

OLD_COLLECTION = "fd_old_model"
NEW_COLLECTION = "fd_new_model"
ALIAS_NAME = "fd_live"  # the alias the application always queries against

client = QdrantClient(":memory:")

CHUNKS = [
    {"text": "Premature withdrawal incurs a 1 percent penalty on the applicable rate.",
     "source": "FD_Policy.json", "doc_type": "policy"},
    {"text": "This penalty does not apply if the FD is closed due to death of the depositor.",
     "source": "FD_Policy.json", "doc_type": "policy"},
    {"text": "Senior citizens receive an additional 0.5 percent interest on all tenures.",
     "source": "FD_Product_Guide.json", "doc_type": "product"},
    {"text": "What is the penalty for early FD closure? A 1 percent penalty applies.",
     "source": "FD_FAQ.json", "doc_type": "faq"},
    {"text": "The FD interest rate for 24 months is 7.1 percent per annum.",
     "source": "FD_Product_Guide.json", "doc_type": "product"},
]


def make_point_id(source: str, index: int) -> int:
    # deterministic ID -- same chunk always gets the same ID
    return int(hashlib.md5(f"{source}::{index}".encode()).hexdigest()[:8], 16)


def build_collection(collection_name: str, model: SentenceTransformer,
                     vector_size: int, chunks: list) -> None:
    """Creates a collection and upserts all chunks embedded with the given model."""

    # create the collection with the specified vector size
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
    )

    # embed all chunks in one batch using the provided model
    texts = [c["text"] for c in chunks]
    embeddings = model.encode(texts, normalize_embeddings=True)

    # build points with deterministic IDs
    points = [
        PointStruct(
            id=make_point_id(chunks[i]["source"], i),
            vector=embeddings[i].tolist(),
            payload={
                "text": chunks[i]["text"],
                "source": chunks[i]["source"],
                "doc_type": chunks[i]["doc_type"],
                # store which model produced these vectors -- key for debugging later
                "embed_model": model._modules['0'].auto_model.config._name_or_path
                if hasattr(model._modules.get('0', None), 'auto_model') else "unknown",
            },
        )
        for i in range(len(chunks))
    ]

    client.upsert(collection_name=collection_name, points=points)
    print(f"Built '{collection_name}' with {len(points)} points "
          f"(vector_size={vector_size})")


def search(collection_name: str, query: str, query_model: SentenceTransformer,
           k: int = 3, label: str = "") -> None:
    """Search a collection using a specific model to embed the query."""

    # embed the query with the provided model
    query_vector = query_model.encode(query, normalize_embeddings=True).tolist()

    results = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=k,
        with_payload=True,
    ).points

    print(f"\n{label}")
    print(f"  Query model : {query_model._first_module().auto_model.config._name_or_path
          if hasattr(query_model._first_module(), 'auto_model') else 'unknown'}")
    print(f"  Collection  : {collection_name}")
    for r in results:
        print(f"  score={r.score:.4f} | {r.payload['text'][:65]}")


def demonstrate_manifest_blind_spot(manifest: dict, current_model: str) -> None:
    """Shows that the manifest as built doesn't detect a model change.
    The fix: store embed_model_name in each manifest entry."""

    print("\n--- Manifest blind spot ---")

    # simulate what the manifest currently tracks (content hash only)
    print("  Current manifest entry (content hash only):")
    print(f"    {{'hash': 'abc123...', 'last_ingested': '2024-01-01'}}")
    print("  After model swap -- manifest still says nothing to ingest:")
    print(f"    content unchanged -> hash unchanged -> 'nothing to ingest'")
    print(f"    but vectors are now stale relative to new model!")

    print("\n  Fixed manifest entry (tracks model name too):")
    print(f"    {{'hash': 'abc123...', 'embed_model': '{current_model}', "
          f"'last_ingested': '2024-01-01'}}")
    print(f"  After model swap to 'all-MiniLM-L6-v2':")
    print(f"    embed_model mismatch detected -> triggers full re-embed")
    print(f"    even though source content (and hash) is unchanged")


# ======================================================================
# Step 1: load both models
# ======================================================================
print("Loading models (may download on first run)...")
old_model = SentenceTransformer(OLD_MODEL_NAME)
new_model = SentenceTransformer(NEW_MODEL_NAME)
print(f"Old model: {OLD_MODEL_NAME} ({OLD_VECTOR_SIZE}-dim)")
print(f"New model: {NEW_MODEL_NAME} ({NEW_VECTOR_SIZE}-dim)")

# ======================================================================
# Step 2: build old collection -- the starting state before migration
# ======================================================================
print("\n--- Step 1: Build collection with OLD model ---")
build_collection(OLD_COLLECTION, old_model, OLD_VECTOR_SIZE, CHUNKS)

# search with old model against old collection -- correct, this works
QUERY = "What is the penalty for closing an FD early?"
search(OLD_COLLECTION, QUERY, old_model,
       label="CORRECT: old model query vs old collection")

# ======================================================================
# Step 3: demonstrate the failure -- search with new model against old
# collection. No error, but scores are meaningless.
# ======================================================================
print("\n--- Step 2: Swap model in code but forget to re-index (THE BUG) ---")
search(OLD_COLLECTION, QUERY, new_model,
       label="WRONG: new model query vs old collection (silent failure)")
print("  No error thrown. Scores look plausible but measure nothing real.")
print("  This is the failure mode -- silent, no exception, wrong answers.")

# ======================================================================
# Step 4: correct migration -- build new collection, validate, swap alias
# ======================================================================
print("\n--- Step 3: Correct migration -- build new collection first ---")
build_collection(NEW_COLLECTION, new_model, NEW_VECTOR_SIZE, CHUNKS)

# validate new collection with new model before swapping
search(NEW_COLLECTION, QUERY, new_model,
       label="VALIDATE: new model query vs new collection (correct)")

# ======================================================================
# Step 5: alias swap -- atomic pointer change, zero downtime
# In-memory mode doesn't support aliases, so we simulate the pattern.
# With Docker/Qdrant Cloud, this would be:
#   client.update_collection_aliases(change_aliases_operations=[
#       CreateAliasOperation(create_alias=CreateAlias(
#           collection_name=NEW_COLLECTION, alias_name=ALIAS_NAME))
#   ])
# ======================================================================
print("\n--- Step 4: Alias swap (simulated -- requires Docker for real aliases) ---")
print(f"  Before swap: alias '{ALIAS_NAME}' -> '{OLD_COLLECTION}'")
print(f"  After swap:  alias '{ALIAS_NAME}' -> '{NEW_COLLECTION}'")
print(f"  This is one API call. Live queries see the new collection instantly.")
print(f"  Old collection stays intact until new one is confirmed working.")
print(f"  Rollback = one API call to point alias back.")

# ======================================================================
# Step 6: manifest blind spot
# ======================================================================
manifest = {
    "FD_Policy.json": {"hash": "abc123", "last_ingested": "2024-01-01"},
}
demonstrate_manifest_blind_spot(manifest, OLD_MODEL_NAME)

# ======================================================================
# Summary
# ======================================================================
print("\n--- Summary ---")
print("  Old collection (old model)  : still exists, serves old traffic via alias")
print("  New collection (new model)  : built, validated, now live via alias")
print("  What changed                : alias pointer (one API call)")
print("  What stayed the same        : all application query code")
print("  Rollback                    : one alias update, instant")
print("  Manifest fix needed         : track embed_model alongside content hash")


Loading models (may download on first run)...
Old model: paraphrase-multilingual-MiniLM-L12-v2 (384-dim)
New model: all-MiniLM-L6-v2 (384-dim)

--- Step 1: Build collection with OLD model ---
Built 'fd_old_model' with 5 points (vector_size=384)

CORRECT: old model query vs old collection
  Query model : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
  Collection  : fd_old_model
  score=0.8909 | What is the penalty for early FD closure? A 1 percent penalty app
  score=0.5689 | Premature withdrawal incurs a 1 percent penalty on the applicable
  score=0.5678 | This penalty does not apply if the FD is closed due to death of t

--- Step 2: Swap model in code but forget to re-index (THE BUG) ---

WRONG: new model query vs old collection (silent failure)
  Query model : sentence-transformers/all-MiniLM-L6-v2
  Collection  : fd_old_model
  score=0.3980 | What is the penalty for early FD closure? A 1 percent penalty app
  score=0.3188 | This penalty does not apply if the FD is clo